In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Libraries loaded successfully
Pandas version: 3.0.2
Numpy version: 2.4.4


In [3]:
import os

# Use absolute path to avoid any working directory issues
DATA_PATH = os.path.expanduser("~/Desktop/europrocure-intelligence/data/raw/export_CAN_2023_2018.csv")

# Confirm the file exists before loading
print(f"File exists: {os.path.exists(DATA_PATH)}")
print(f"File size: {os.path.getsize(DATA_PATH) / (1024**3):.2f} GB")

# Read just the first 1000 rows to inspect structure
df_sample = pd.read_csv(
    DATA_PATH,
    nrows=1000,
    encoding='utf-8',
    low_memory=False
)

print(f"\nSample shape: {df_sample.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df_sample.columns, 1):
    print(f"  {i:02d}. {col}")

File exists: True
File size: 3.73 GB

Sample shape: (1000, 75)

Column names:
  01. ID_NOTICE_CAN
  02. TED_NOTICE_URL
  03. YEAR
  04. ID_TYPE
  05. DT_DISPATCH
  06. XSD_VERSION
  07. CANCELLED
  08. CORRECTIONS
  09. B_MULTIPLE_CAE
  10. CAE_NAME
  11. CAE_NATIONALID
  12. CAE_ADDRESS
  13. CAE_TOWN
  14. CAE_POSTAL_CODE
  15. CAE_GPA_ANNEX
  16. ISO_COUNTRY_CODE
  17. ISO_COUNTRY_CODE_GPA
  18. B_MULTIPLE_COUNTRY
  19. ISO_COUNTRY_CODE_ALL
  20. CAE_TYPE
  21. EU_INST_CODE
  22. MAIN_ACTIVITY
  23. B_ON_BEHALF
  24. B_INVOLVES_JOINT_PROCUREMENT
  25. B_AWARDED_BY_CENTRAL_BODY
  26. TYPE_OF_CONTRACT
  27. TAL_LOCATION_NUTS
  28. B_FRA_AGREEMENT
  29. FRA_ESTIMATED
  30. B_FRA_CONTRACT
  31. B_DYN_PURCH_SYST
  32. CPV
  33. MAIN_CPV_CODE_GPA
  34. ID_LOT
  35. ADDITIONAL_CPVS
  36. B_GPA
  37. GPA_COVERAGE
  38. LOTS_NUMBER
  39. VALUE_EURO
  40. VALUE_EURO_FIN_1
  41. VALUE_EURO_FIN_2
  42. B_EU_FUNDS
  43. TOP_TYPE
  44. B_ACCELERATED
  45. OUT_OF_DIRECTIVES
  46. CRIT_CODE
  47. C

In [4]:
# Check data types and non-null counts for all 75 columns
print(f"Dataset shape: {df_sample.shape}")
print(f"\nColumn details:")
print(df_sample.dtypes.to_string())

Dataset shape: (1000, 75)

Column details:
ID_NOTICE_CAN                     int64
TED_NOTICE_URL                      str
YEAR                              int64
ID_TYPE                           int64
DT_DISPATCH                         str
XSD_VERSION                         str
CANCELLED                         int64
CORRECTIONS                       int64
B_MULTIPLE_CAE                      str
CAE_NAME                            str
CAE_NATIONALID                      str
CAE_ADDRESS                         str
CAE_TOWN                            str
CAE_POSTAL_CODE                     str
CAE_GPA_ANNEX                       str
ISO_COUNTRY_CODE                    str
ISO_COUNTRY_CODE_GPA                str
B_MULTIPLE_COUNTRY                  str
ISO_COUNTRY_CODE_ALL            float64
CAE_TYPE                            str
EU_INST_CODE                        str
MAIN_ACTIVITY                       str
B_ON_BEHALF                         str
B_INVOLVES_JOINT_PROCUREMENT        s

In [5]:
# Check missingness across all 75 columns
missing = df_sample.isnull().sum()
missing_pct = (df_sample.isnull().sum() / len(df_sample) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct,
    'dtype': df_sample.dtypes
}).sort_values('missing_pct', ascending=False)

print("Missingness report (all 75 columns):")
print(missing_df.to_string())

Missingness report (all 75 columns):
                              missing_count  missing_pct    dtype
ISO_COUNTRY_CODE_ALL                   1000       100.00  float64
EU_INST_CODE                            970        97.00      str
B_ACCELERATED                           947        94.70      str
FRA_ESTIMATED                           890        89.00      str
INFO_ON_NON_AWARD                       887        88.70      str
CRIT_PRICE_WEIGHT                       722        72.20      str
CRIT_CRITERIA                           626        62.60      str
CRIT_WEIGHTS                            626        62.60      str
WIN_NATIONALID                          611        61.10      str
NUMBER_TENDERS_NON_EU                   540        54.00  float64
NUMBER_TENDERS_OTHER_EU                 538        53.80  float64
AWARD_EST_VALUE_EURO                    518        51.80  float64
CONTRACT_NUMBER                         513        51.30      str
NUMBER_TENDERS_SME                     

In [6]:
# Focus on the value columns — these are critical for spend analysis
value_cols = [
    'VALUE_EURO',
    'VALUE_EURO_FIN_1',
    'VALUE_EURO_FIN_2',
    'AWARD_VALUE_EURO',
    'AWARD_VALUE_EURO_FIN_1',
    'AWARD_EST_VALUE_EURO'
]

print("Value column analysis:")
print("=" * 60)
for col in value_cols:
    non_null = df_sample[col].notna().sum()
    coverage = (non_null / len(df_sample) * 100)
    if non_null > 0:
        median = df_sample[col].median()
        mean = df_sample[col].mean()
        max_val = df_sample[col].max()
        print(f"\n{col}")
        print(f"  Coverage : {coverage:.1f}% ({non_null}/1000 rows)")
        print(f"  Median   : €{median:,.0f}")
        print(f"  Mean     : €{mean:,.0f}")
        print(f"  Max      : €{max_val:,.0f}")
    else:
        print(f"\n{col}")
        print(f"  Coverage : {coverage:.1f}% — completely empty in sample")

Value column analysis:

VALUE_EURO
  Coverage : 96.8% (968/1000 rows)
  Median   : €1,717,028
  Mean     : €11,724,174
  Max      : €120,979,952

VALUE_EURO_FIN_1
  Coverage : 96.8% (968/1000 rows)
  Median   : €1,642,719
  Mean     : €11,420,692
  Max      : €120,979,952

VALUE_EURO_FIN_2
  Coverage : 96.8% (968/1000 rows)
  Median   : €1,642,719
  Mean     : €11,420,692
  Max      : €120,979,952

AWARD_VALUE_EURO
  Coverage : 83.1% (831/1000 rows)
  Median   : €56,040
  Mean     : €1,094,654
  Max      : €120,979,952

AWARD_VALUE_EURO_FIN_1
  Coverage : 94.4% (944/1000 rows)
  Median   : €36,045
  Mean     : €963,620
  Max      : €120,979,952

AWARD_EST_VALUE_EURO
  Coverage : 48.2% (482/1000 rows)
  Median   : €70,000
  Mean     : €1,086,634
  Max      : €89,593,020


In [7]:
# Analyse key categorical columns
cat_cols = {
    'YEAR': 'Year of publication',
    'TOP_TYPE': 'Procedure type',
    'TYPE_OF_CONTRACT': 'Contract type (W/S/U)',
    'ISO_COUNTRY_CODE': 'Buyer country',
    'CAE_TYPE': 'Contracting authority type',
    'B_FRA_AGREEMENT': 'Framework agreement',
    'B_CONTRACTOR_SME': 'SME contractor',
    'CRIT_CODE': 'Award criteria'
}

print("Categorical column distributions:")
print("=" * 60)
for col, description in cat_cols.items():
    print(f"\n{col} — {description}")
    print(f"  Unique values : {df_sample[col].nunique()}")
    print(f"  Top values    :")
    top = df_sample[col].value_counts().head(5)
    for val, count in top.items():
        pct = count / len(df_sample) * 100
        print(f"    {str(val):<20} {count:>5} rows  ({pct:.1f}%)")

Categorical column distributions:

YEAR — Year of publication
  Unique values : 6
  Top values    :
    2018                   401 rows  (40.1%)
    2023                   323 rows  (32.3%)
    2022                   141 rows  (14.1%)
    2020                    61 rows  (6.1%)
    2019                    44 rows  (4.4%)

TOP_TYPE — Procedure type
  Unique values : 6
  Top values    :
    OPE                    891 rows  (89.1%)
    RES                     53 rows  (5.3%)
    AWP                     22 rows  (2.2%)
    NOC                     20 rows  (2.0%)
    NIC                     11 rows  (1.1%)

TYPE_OF_CONTRACT — Contract type (W/S/U)
  Unique values : 3
  Top values    :
    U                      476 rows  (47.6%)
    W                      339 rows  (33.9%)
    S                      185 rows  (18.5%)

ISO_COUNTRY_CODE — Buyer country
  Unique values : 24
  Top values    :
    SI                     209 rows  (20.9%)
    FR                     188 rows  (18.8%)
    PL       

In [8]:
# Check date columns and CPV codes
print("Date column samples:")
print("=" * 60)
print(f"\nDT_DISPATCH sample values:")
print(df_sample['DT_DISPATCH'].dropna().head(5).tolist())

print(f"\nDT_AWARD sample values:")
print(df_sample['DT_AWARD'].dropna().head(5).tolist())

print(f"\nCPV column analysis:")
print(f"  Unique CPV codes    : {df_sample['CPV'].nunique()}")
print(f"  Sample CPV values   : {df_sample['CPV'].dropna().head(10).tolist()}")
print(f"  CPV missing         : {df_sample['CPV'].isnull().sum()} rows")

print(f"\nWIN_COUNTRY_CODE (vendor country):")
print(df_sample['WIN_COUNTRY_CODE'].value_counts().head(10).to_string())

Date column samples:

DT_DISPATCH sample values:
['22/12/17', '22/12/17', '22/12/17', '21/12/18', '21/12/18']

DT_AWARD sample values:
['18/12/17', '18/12/17', '18/12/18', '13/07/18', '07/12/20']

CPV column analysis:
  Unique CPV codes    : 168
  Sample CPV values   : [72300000, 79411000, 79411000, 79000000, 79000000, 72000000, 72000000, 72000000, 64200000, 64200000]
  CPV missing         : 0 rows

WIN_COUNTRY_CODE (vendor country):
WIN_COUNTRY_CODE
SI         190
PL         151
FR         143
DE          98
RO          50
LT          49
ES          30
PT---PT     17
PL---PL     16
BG          12


In [9]:
# Check vendor names and notice level duplication
print("Vendor name analysis:")
print("=" * 60)
print(f"\nWIN_NAME missing        : {df_sample['WIN_NAME'].isnull().sum()} rows ({df_sample['WIN_NAME'].isnull().mean()*100:.1f}%)")
print(f"Unique vendor names     : {df_sample['WIN_NAME'].nunique()}")
print(f"\nTop 10 vendors by contract count:")
print(df_sample['WIN_NAME'].value_counts().head(10).to_string())

print(f"\nNotice level duplication:")
print(f"Total rows              : {len(df_sample)}")
print(f"Unique ID_NOTICE_CAN    : {df_sample['ID_NOTICE_CAN'].nunique()}")
print(f"Unique ID_AWARD         : {df_sample['ID_AWARD'].nunique()}")
duplication_ratio = len(df_sample) / df_sample['ID_NOTICE_CAN'].nunique()
print(f"Avg rows per notice     : {duplication_ratio:.2f}")

Vendor name analysis:

WIN_NAME missing        : 166 rows (16.6%)
Unique vendor names     : 471

Top 10 vendors by contract count:
WIN_NAME
SALUS, Veletrgovina, družba za promet s farmacevtskimi, medicinskimi in drugimi proizvodi, d.o.o.    61
Kemofarmacija, veletrgovina za oskrbo zdravstva, d.d.                                                39
GOPHARM trgovina in laboratorijske storitve, d.o.o.                                                  19
SANOLABOR, podjetje za prodajo medicinskih, laboratorijskih in farmacevtskih proizvodov, d.d.        16
Silvijus Abramavičius                                                                                15
ZAVOD REPUBLIKE SLOVENIJE ZA TRANSFUZIJSKO medicino                                                  15
LENIS farmacevtika d.o.o.                                                                            12
Stryker Polska Sp. z o.o.                                                                            11
Johnson & Johnson Poland Sp.

# Data Profiling Summary

**Dataset:** TED Contract Award Notices 2018-2023  
**File size:** 3.73 GB  
**Total columns:** 75  

## Key Findings

1. VALUE_EURO_FIN_2 has 96.8% coverage — primary value column confirmed
2. Mean (€11.7M) >> Median (€1.6M) — large outliers present, need capping
3. DT_DISPATCH and DT_AWARD format is DD/MM/YY — needs datetime parsing
4. CPV codes are integers, 0% missing — extract first 2 digits for category
5. WIN_COUNTRY_CODE has dirty values (PT---PT) — take first value before ---
6. WIN_NAME is 16.6% missing — acceptable for vendor analysis
7. Avg 4.02 rows per notice — must deduplicate on ID_NOTICE_CAN for counts
8. TOP_TYPE is 89.1% open procedure in sample — COVID impact hypothesis testable
9. ISO_COUNTRY_CODE_ALL is 100% missing — drop this column
10. 6 columns above 70% missing — drop candidates confirmed

## Columns to Drop (>70% missing)

- ISO_COUNTRY_CODE_ALL (100%)
- EU_INST_CODE (97%)
- B_ACCELERATED (94.7%)
- FRA_ESTIMATED (89%)
- INFO_ON_NON_AWARD (88.7%)
- CRIT_PRICE_WEIGHT (72.2%)

## Decisions

- **Primary value column:** VALUE_EURO_FIN_2
- **Date format:** DD/MM/YY
- **Deduplication key:** ID_NOTICE_CAN